## Distant Supervision

To run this script, you need the following files found in the /data directory:
- "news_headlines_usa_neutral.csv"
- "news_headlines_usa_biased.csv"

In [1]:
!pip install transformers

In [2]:
!pip install --upgrade tensorflow transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.3/615.3 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.6 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.1
    Uninstalling tensorboard-2.17.1:
      Successfully uninstalled tensorboard-2.17.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.3
    Uninstalling transformers-4.46.3:
      Successfully uninstalled transformers-4.46.3
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2

In [17]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [3]:
# gpu card
!nvidia-smi

Wed Dec 11 09:43:10 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L4                      Off | 00000000:00:03.0 Off |                    0 |
| N/A   36C    P8              12W /  72W |      1MiB / 23034MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available:  1


In [ ]:
# Resolve paths relative to the repository root so this notebook runs from the repo
# root, from notebooks/, or on Colab after cloning the repo:
#   !git clone https://github.com/yasmeenhany/media-bias-detection.git
#   %cd media-bias-detection
import os
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
os.chdir(REPO_ROOT)
print("Working directory:", REPO_ROOT)

In [6]:
import time
import random
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import tensorflow as tf
from transformers import RobertaTokenizer, TFRobertaForSequenceClassification
from transformers import BertTokenizer, TFBertForSequenceClassification, TFDistilBertForSequenceClassification, DistilBertTokenizer

In [7]:
# set seed, TF uses python ramdom and numpy library, so these must also be fixed
tf.random.set_seed(0)
random.seed(0)
np.random.seed(0)
os.environ['PYTHONHASHSEED']=str(0)
os.environ['TF_DETERMINISTIC_OPS'] = '0'

## Media Cloud

The idea is that in outlets such as Alternet, Breitbart, Federalist the density of bias is higher than in Reuters. Using Media Cloud, I extract article headers of these outlets covering controversial topics and assume them to be biased in the case of Breitbart etc.

In [27]:
def read_media_cloud_data(path, label):
    """Read in data downloaded from media cloud and assign a label to all rows"""
    df = pd.read_csv(path)
    df['Label_bias'] = label
    df = df.rename({'title': 'sentence'}, axis=1)
    return df

# read in two datasets
PATH_biased = "data/news_headlines_usa_biased.csv"
PATH_neutral = "data/news_headlines_usa_neutral.csv"
df_biased = read_media_cloud_data(PATH_biased, 1)
df_neutral = read_media_cloud_data(PATH_neutral, 0)

# combine them
df_distant = pd.concat([df_biased,df_neutral], axis=0, ignore_index=1)
df_distant = shuffle(df_distant)

# train-test split
df_distant_train, df_distant_test = train_test_split(df_distant, test_size=0.2)

In [28]:
df_distant['Label_bias'].value_counts()

,count
Label_bias,
0,83143
1,45605


## Political Social Media Posts

In [8]:
def read_social_media_data(path):
    """Read in data and assign a label to all rows"""
    df = pd.read_csv(path, encoding="Latin1")
    df['Label_bias'] = df["bias"].map({"partisan": 1, "neutral": 0})
    df = df.rename({'text': 'sentence'}, axis=1)
    return df

# read in two datasets
PATH = "data/political_social_media.csv"
df_distant = read_social_media_data(PATH)

df_distant = shuffle(df_distant)

# train-test split
df_distant_train, df_distant_test = train_test_split(df_distant, test_size=0.2)

In [9]:
df_distant['Label_bias'].value_counts()

,count
Label_bias,
0,3689
1,1311


## Train

In [10]:
# from transformers import AutoTokenizer
# def preprocess(df):
#     """convert a pandas dataframe into a tensorflow dataset"""
#     target = df.pop('Label_bias')
#     sentence = df.pop('sentence')

#     #tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased') #uncased
#     #tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
#     # tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
#     tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small", use_fast=False)

#     train_encodings = tokenizer(
#                         sentence.tolist(),
#                         add_special_tokens = True, # add [CLS], [SEP]
#                         truncation = True, # cut off at max length of the text that can go to BERT
#                         padding = True, # add [PAD] tokens
#                         return_attention_mask = True, # add attention mask to not focus on pad tokens
#                         max_length=128
#               )

#     dataset = tf.data.Dataset.from_tensor_slices(
#         (dict(train_encodings),
#          target.tolist()))
#     return dataset

In [20]:
# from transformers import T5Tokenizer, TFT5ForConditionalGeneration, T5ForConditionalGeneration
# def preprocess(data):
#     # Tokenize input sentences with T5 prompt
#     tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
#     inputs = tokenizer(data["sentence"].apply(
#         lambda x: "" + x).tolist(),
#         max_length=128, padding="max_length", truncation=True, return_tensors="tf")

#     # Map the labels to the desired format (one-hot encoding)
#     label_map = {0: [1, 0], 1: [0, 1]}  # 0 -> Non-biased -> [1, 0], 1 -> Biased -> [0, 1]
#     one_hot_labels = data["Label_bias"].map(label_map).tolist()
#     targets = tf.convert_to_tensor(one_hot_labels, dtype=tf.int32)  # Convert to tensor
#     # Create TensorFlow dataset
#     dataset = tf.data.Dataset.from_tensor_slices((
#         {"input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"]},
#         targets
#     ))

#     return dataset

In [11]:
# # pandas -> tensorflow
# train_distant_dataset = preprocess(df_distant_train)
# test_distant_dataset = preprocess(df_distant_test)

# # batch and randomize
# BUFFER_SIZE = 10000
# BATCH_SIZE = 64

# train_distant_dataset = train_distant_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.experimental.AUTOTUNE)
# test_distant_dataset = test_distant_dataset.batch(BATCH_SIZE).prefetch(tf.data.experimental.AUTOTUNE)

CPU: 64 mini-batch size has 19hours ETA. 256 mini-batch size yields 16hours ETA

GPU: 256 mini-batch size yields 20 mins ETA

In [26]:
tf.keras.backend.clear_session()

In [31]:

# # class T5WithClassificationHead(tf.keras.Model):
# #     def __init__(self, base_model, num_classes):
# #         super(T5WithClassificationHead, self).__init__()
# #         self.base_model = base_model
# #         self.num_classes = num_classes
# #         self.classifier = tf.keras.layers.Dense(num_classes, activation="softmax")

# #     def call(self, input_ids, attention_mask=None):
# #         encoder_outputs = self.base_model.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]

# #         pooled_output = tf.reduce_mean(encoder_outputs, axis=1)

# #         logits = self.classifier(pooled_output)
# #         return logits
# # train entire model with distant signals
# # bert = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased") #DistilBert Model transformer with a sequence classification/regression head on top (a linear layer on top of the pooled output
# #bert = TFBertForSequenceClassification.from_pretrained("bert-base-uncased")
# # roberta = TFRobertaForSequenceClassification.from_pretrained('roberta-base')
# # t5 = TFT5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
# # t5 = T5WithClassificationHead(t5, 2)
# from transformers import AutoModelForSequenceClassification
# deberta =   AutoModelForSequenceClassification.from_pretrained(
#                 "microsoft/deberta-v3-small"
#             )

# callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True) # after 3 epochs without improvement, stop training
# optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
# deberta.compile(optimizer=optimizer, loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False))

In [31]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
import torch

from sklearn.model_selection import train_test_split

def run_model_single_split(model_name="microsoft/deberta-v3-small", freeze_encoder=False, pretrained=False, plot=False, tuning=False):


    train_dataset = df_distant_train
    val_dataset = df_distant_test

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

    def tokenize_data(data):
        return tokenizer(
            data['sentence'].tolist(),
            padding=True,
            truncation=True,
            max_length=150,
            return_tensors="pt"
        )

    train_encodings = tokenize_data(train_dataset)
    val_encodings = tokenize_data(val_dataset)

    train_labels = torch.tensor(train_dataset['Label_bias'].tolist())
    val_labels = torch.tensor(val_dataset['Label_bias'].tolist())

    train_data = Dataset.from_dict({
        'input_ids': train_encodings['input_ids'].numpy(),
        'attention_mask': train_encodings['attention_mask'].numpy(),
        'labels': train_labels.numpy()
    })

    val_data = Dataset.from_dict({
        'input_ids': val_encodings['input_ids'].numpy(),
        'attention_mask': val_encodings['attention_mask'].numpy(),
        'labels': val_labels.numpy()
    })

    def model_init():
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name
        )
        if freeze_encoder:
            for param in model.base_model.parameters():
                param.requires_grad = False
        return model

    training_args = TrainingArguments(
        output_dir='./results/single_split',
        evaluation_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=1,
        weight_decay=0.1,
        seed=42
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
        f1_micro = precision_recall_fscore_support(labels, preds, average='micro')[2]
        f1_macro = precision_recall_fscore_support(labels, preds, average='macro')[2]

        acc = accuracy_score(labels, preds)
        return {
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'f1_micro': f1_micro,
            'f1_macro': f1_macro
        }

    if tuning:
        def hyperparameter_search_space(trial):
            return {
                "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
                "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
                "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 5),
                "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
            }

        trainer = Trainer(
            model_init=model_init,
            args=training_args,
            train_dataset=train_data,
            eval_dataset=val_data,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics
        )

        best_run = trainer.hyperparameter_search(
            direction="maximize",
            n_trials=45,
            compute_objective=lambda metrics: metrics["eval_f1"],
            hp_space=hyperparameter_search_space
        )

        print(f"Best Hyperparameters: {best_run.hyperparameters}")
        for key, value in best_run.hyperparameters.items():
            setattr(training_args, key, value)

    else:
        trainer = Trainer(
            model_init=model_init,
            args=training_args,
            train_dataset=train_data,
            eval_dataset=val_data,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics
        )

    trainer.train()

    eval_result = trainer.evaluate()
    val_loss = eval_result.get('eval_loss', None)
    val_acc = eval_result.get('eval_accuracy', None)
    val_prec = eval_result.get('eval_precision', None)
    val_rec = eval_result.get('eval_recall', None)
    val_f1 = eval_result.get('eval_f1', None)
    val_f1_micro = eval_result.get('eval_f1_micro', None)
    val_f1_macro = eval_result.get('eval_f1_macro', None)

    return trainer.model, val_loss, val_acc, val_prec, val_rec, val_f1, val_f1_micro, val_f1_macro



In [32]:
# history_t5 = deberta.fit(train_distant_dataset, epochs=2, validation_data = test_distant_dataset, callbacks=[callback])
# history_deberta = trainer.train()
import os
os.environ["WANDB_DISABLED"] = "true"

trained_model, val_loss, val_acc, val_prec, val_rec, val_f1, val_f1_micro, val_f1_wmacro = run_model_single_split(freeze_encoder=False, pretrained=False, tuning=False)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-31-28afdc062e57>:112: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSeq

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Micro,F1 Macro
1,0.160900,0.154129,0.943262,0.943941,0.943262,0.943464,0.943262,0.938414


In [34]:
# trained_model.save_pretrained("./checkpoints/deberta_social_media")
import torch
# torch.save(trained_model.state_dict(), "./deberta_social_media_model_weights.pth")
torch.save(trained_model.state_dict(), "./deberta_news_weights.pth")

In [26]:
# trained_layer = t5.get_layer(index=0).get_weights()
# # roberta.save_weights('./checkpoints/roberta_final_checkpoint_news_headlines_USA')
# # t5.save_weights('./checkpoints/t5_final_checkpoint_social_media')
# t5.save_weights('./checkpoints/t5_final_checkpoint_news_headlines_USA')


# #bert.load_weights('./checkpoints/final_checkpoint_distant_learning')

In [ ]:
len(trained_layer)

282